<a href="https://colab.research.google.com/github/bitsteller/dataanalys-for-kollektivtrafik/blob/main/Kod/Laboration%203/Uppgift2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboration 3 – Uppgift 2

Den här uppgiften handlar om att bearbeta data med **Polars**.
I uppgiften tittar vi på data om tåg som har passerat Hässleholm C under 2025. Uppgiften ska ta cirka 30min.

## Jobba så här
1. Kör exempelcellen i varje avsnitt.
2. Fyll i kod i övningscellen under.
3. Kör övningscellen och kontrollera att resultatet ser rimligt ut.

## Om datasetet

Datasetet innehåller information alla genomgående tåg (som har både ankomst och avgång) som har passerat Hässleholm under 2025.
Mer detaljerad information om datasetet finns i labbinstruktionen.


## 1) Läsa in data med polars

Vi börjar med att läsa in `trains.parquet`. En parquet-fil innehåller data i tabellformat (ungefär som en Excel eller CSV).

* `pl.read_parquet()` läser in en parquet-fil som en tabell (kallas också DataFrame)
* `.height` anger antalet rader
* `.columns` listar namnen på alla kolumner

In [ ]:
import polars as pl

BASE_URL = "https://raw.githubusercontent.com/bitsteller/dataanalys-for-kollektivtrafik/main/Data/Laboration%203"
TRAINS_URL = f"{BASE_URL}/trains.parquet"

trains = pl.read_parquet(TRAINS_URL)
print("Antal rader i trains:", trains.height)
print("Kolumner:", trains.columns)


In [ ]:
# Övning: Läs in locations.parquet som finns på URL:n `LOCATIONS_URL` och skriv ut antal rader + kolumnnamn.
LOCATIONS_URL = f"{BASE_URL}/locations.parquet"

locations = None  # byt ut None mot ett kommando som läser in locations
print("Antal rader i locations:", None)  # byt ut None
print("Kolumner:", None)  # byt ut None


## 2) Inspektera datasetet

Ett enkelt sätt att inspektera tabellen är att titta på de första raderna med funktionen `head()`.

* `head()` visar de första raderna i en tabell
* `select()` kan användas för att behålla endast vissa kolumner

In [ ]:
trains.head()

In [ ]:
# Övning: Visa de första raderna på `trains_selected` som endast innehåller kolumnerna `Date`, `Weekday` och `TrainOwner`.
trains_selected = trains.select(["Date", "Weekday", "TrainOwner"])  # byt gärna kolumner

#Använd funktionen head() här


In [ ]:
# Bonusövning: Funktionen describe() kan användas på en tabell för få sammanfattande statistik som medelvärde och percentiler
# Testa lägga till fler numeriska kolumner (Exempel: "Hour", "PlannedDwellTime", "ActualDwellTime")

trains.select(["Hour"]).describe()

## 3) Filtrera data

Nu filtrerar vi fram en delmängd för att se hur många av tågen i datasetet har kört på vardagar.

* `filter()` används för att filtrerar raderna där ett viist kriterium är sant
* `pl.col()` används för att referera till en kolumn
* `is_in()` är sant om värdet för en kolumn finns i den angivna listan, annars falskt
* `&` används för att kombinera kriterer där alla måste vara sanna för att raden ska vara kvar
* `==`, `<`, `>=`, osv. kan användas för jämförelser

In [ ]:
vardag = trains.filter(
    (pl.col("Weekday").is_in(["Mon", "Tue", "Wed", "Thu", "Fri"]))
)

print("Antal tåg vardagar:", vardag.height)


Antal tåg vardagar: 28497


In [ ]:
# Övning: Beräkna antal tåg som har gått på helger (lör, sön)





In [ ]:
# Bonusövning: Beräkna antal tåg som har gått mellan klockan 6 och 9 under vardagar genom att kombinera flera kriterier.

morgon_vardag = trains.filter(
    (pl.col("Weekday").is_in(["Mon", "Tue", "Wed", "Thu", "Fri"]))
    & True #ersätt True med ett kriterium som filtrerar tåg som går klockan 6 eller senare
    & (pl.col("Hour") < 9)
)

print("Rader morgon vardag:", morgon_vardag.height)

Rader morgon vardag: 6428


## 4) Gruppera data

Vi kan också grupperar data för att se antalet tåg för varje veckodag.

* `group_by()` grupperar alla rader med samma värde i den angivna kolumnen
* `agg()` används för att räkna ut mått för varje grupp
* `len()` räknar ut antalet per grupp
* `alias()` ändrar namnet på en kolumn
* `sort()` sorterar tabellen enligt en eller flera kolumner


In [ ]:
per_veckodag = (
    trains.group_by("Weekday")
    .agg(pl.len().alias("Antal"))
    .sort("Antal", descending=True)
)

per_veckodag


In [ ]:
# Övning: Gruppera på ProductName för att se antalet tåg per typ av tåg




## 5) Aggregation

Istället för antalet kan vi även använda andra mått för att agreggera.

* `median()` beräknar medianen
* `std()` beräknar standardavvikelsen

In [ ]:
agg_per_dag = (
    trains.group_by("ProductName")
    .agg([
        pl.len().alias("AntalTag"),
        pl.col("ActualDwellTime").median().alias("MedianActualDwell"),
        pl.col("ActualDwellTime").std().alias("StdActualDwell")
    ])
    .sort("MedianActualDwell", descending=True)
)

agg_per_dag


In [ ]:
# Övning: Testa att göra samma aggregering för Weekday istället.


